In [2]:
%load_ext autoreload
%autoreload 2

from setup_imports import *  # noqa: F401,F403

# Tag phrases from text, then sync tags to Anki

Walks through the real end-to-end workflow with one concrete example:

1. `add_tags_from_text` — from a piece of target-language text, find the minimum set of *existing* phrases whose translation already covers its vocab, and tag those phrases in Firestore (tag stored **unprefixed**).
2. Confirm the tag landed on the phrase's translation in Firestore.
3. `sync_tag_to_anki` — sync that tag into the live Anki collection (`dry_run=True` first). It should show up on the note as `fs::food_and_drink` — the `fs::` prefix is added only at this step; Firestore itself keeps the bare `food_and_drink`.
4. Only once you're happy with the dry-run report: flip `RUN_FOR_REAL` to `True` and re-run the last cell to actually write to Anki.

In [3]:
from phrases.search import add_tags_from_text
from connections.anki_collection import get_anki_collection, close_anki_collection
from anki_sync import sync_tag_to_anki

## Config

In [15]:
#load some text

with open("../data/text_to_process/wildfires.txt", "r", encoding="utf-8") as f:
    text = f.readlines()

text = " ".join([line.strip() for line in text if line.strip()])

In [16]:
text

'Det är stora bränder i Spanien och i Frankrike. Många tusen människor har tvingats fly i länderna. Folk i området Cap Ferret i sydvästra Frankrike har fått hjälp att fly i båt. Människor berättar att aska blåser in över stränderna. Elden sprider sig snabbt. Personalen som jobbar med att släcka har inte kontroll över elden. Både Spanien och Frankrike ska få hjälp från EU att släcka och stoppa elden. Flygplan från flera länder ska hjälpa Frankrike. De ska släcka bränderna i skogen från luften. Sverige ska också skicka två flygplan.'

In [ ]:
TEXT = text # Swedish for "a bottle of wine"
LANGUAGE = "sv-SE"
TAG = "food_and_drink"
SOURCE_LANGUAGE = "en-GB"

# Safety gate for the last cell - real Anki write only happens if True
RUN_FOR_REAL = True

## 1. Tag the covering phrase(s) in Firestore

Finds the minimum set of existing phrases whose Swedish translation covers the vocab in `TEXT`, and tags them. This is a real Firestore write (low-risk/reversible - see `delete_tag_from_firestore` in `phrases/search.py` if you need to undo it).

In [5]:
tagged_phrases, missing = add_tags_from_text(TEXT, LANGUAGE, TAG)

print(f"\nTagged {len(tagged_phrases)} phrase(s):")
for p in tagged_phrases:
    sv_text = p.translations[LANGUAGE].text
    print(f"  {p.key} | en: {p.english!r} | sv-SE: {sv_text!r}")
print(f"Missing vocab (no covering phrase found): {missing}")

(y) Authenticated with Google Cloud project: swedish-course
2026-07-26 13:13:48 - audio-language-trainer - INFO - nlp.py:88 - _load_spacy_model: loaded spaCy model 'sv_core_news_lg' for language 'sv'
add_tags_from_text: 2/2 words covered by 1 phrase(s); 1 tokens ignored (not a content word or no Wiktionary entry); 0 words missing (no matching phrase found)
  Ignored tokens: ['en']

Tagged 1 phrase(s):
  a_bottle_of_wine_d1ae8f | en: 'A bottle of wine' | sv-SE: 'En flaska vin'
Missing vocab (no covering phrase found): {'verbs': [], 'vocab': []}


## 2. Confirm the tag in Firestore, unprefixed

In [6]:
for p in tagged_phrases:
    print(p.key, "->", p.translations[LANGUAGE].tags)

a_bottle_of_wine_d1ae8f -> ['SURVIVAL', 'Pack01', 'food_and_drink']


## 3. Sync the tag into the live Anki collection (dry run)

`dry_run=True` makes zero writes anywhere - safe to re-run as many times as you like.

In [7]:
col = get_anki_collection()
try:
    report = sync_tag_to_anki(col, TAG, SOURCE_LANGUAGE, LANGUAGE, dry_run=False)
    print(report.summary())
    for r in report.results:
        print(" ", r)
finally:
    close_anki_collection()

2026-07-26 13:14:39 - audio-language-trainer - INFO - anki_sync.py:76 - Duplicate (SourceText, TargetText) while indexing notes: ('Yes', 'Ja') -> note ids 1769962030920 and 1770057590273
2026-07-26 13:14:39 - audio-language-trainer - INFO - anki_sync.py:76 - Duplicate (SourceText, TargetText) while indexing notes: ('A high rate of interest', 'En hög ränta') -> note ids 1775983052551 and 1784627291433
2026-07-26 13:14:39 - audio-language-trainer - INFO - anki_sync.py:76 - Duplicate (SourceText, TargetText) while indexing notes: ('Did they have sex?', 'Hade de sex?') -> note ids 1775983052555 and 1784627291437
2026-07-26 13:14:39 - audio-language-trainer - INFO - anki_sync.py:76 - Duplicate (SourceText, TargetText) while indexing notes: ('Did they give a reason?', 'Sa de varför?') -> note ids 1775983052559 and 1784627291441
2026-07-26 13:14:39 - audio-language-trainer - INFO - anki_sync.py:76 - Duplicate (SourceText, TargetText) while indexing notes: ('That sounds good', 'Det låter bra')

Syncing phrases to Anki: 100%|██████████| 1/1 [00:00<00:00, 86.56it/s]

Synced 1 phrase(s):
  0 note(s) created
  1 guid(s) healed
  1 note(s) have tags updated
  SyncResult(phrase_key='a_bottle_of_wine_d1ae8f', created=False, guid_healed=True, tags_changed=True, error=None)


## 4. Run for real — only after reviewing the dry-run report above

Close Anki Desktop first (it holds an exclusive lock on the collection file). Set `RUN_FOR_REAL = True` in the config cell and re-run it, then run this cell. Takes a real Anki backup before writing, same as `scripts/sync_anki_tags.py`.

In [7]:
import os
from pathlib import Path

if not RUN_FOR_REAL:
    print("Skipped - set RUN_FOR_REAL = True in the config cell above and re-run both cells when ready.")
else:
    col = get_anki_collection()
    try:
        backup_folder = str(Path(os.environ["ANKI_COLLECTION_PATH"]).parent / "backups")
        os.makedirs(backup_folder, exist_ok=True)
        col.create_backup(backup_folder=backup_folder, force=True, wait_for_completion=True)

        report = sync_tag_to_anki(col, TAG, SOURCE_LANGUAGE, LANGUAGE, dry_run=False)
        print(report.summary())
        for r in report.results:
            print(" ", r)
    finally:
        close_anki_collection()

Skipped - set RUN_FOR_REAL = True in the config cell above and re-run both cells when ready.
